# tiny-voca in Spanish: the bilingual voice (tiny-speech)

One small voice (tiny-voca, about 8 MB in the browser) that speaks Mexican Spanish and English, for MapSMA's answers. Code and
details: `tiny-speech/training/` in github.com/larsboesen-prog/ranked-feed (branch `claude/tiny-speech-bilingual-v7lyp0`):
`RECIPE.md`, `gpu/README.md`.

**MODE** (next cell): `"design"` (GPU, about 20 minutes): a few Mexican voices designed from descriptions read the site's
sentences in both languages, for the owner to pick one. Later modes read the training sentences in the picked voice and train.
Kaggle: Accelerator **GPU T4**, Internet **on**, then **Save Version → Save & Run All**.

In [ ]:
MODE = "design"   # "design" (GPU): candidate voices and their samples
import os, sys, shutil, pathlib, json, subprocess, socket


def run(*cmd):
    """a command, its output shown as it comes; the notebook stops with an error if it fails"""
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"failed ({p.returncode}): {' '.join(map(str, cmd))}")


try:
    socket.getaddrinfo("github.com", 443)
    socket.getaddrinfo("huggingface.co", 443)
except OSError:
    raise RuntimeError("No internet in this session. Kaggle: Settings → Internet on (the account must be phone-verified).")
KAGGLE = os.path.exists("/kaggle/working")
OUT = pathlib.Path("/kaggle/working/tiny-voice") if KAGGLE else pathlib.Path("tiny-voice").resolve()
BASE = pathlib.Path("/kaggle/tmp") if KAGGLE else pathlib.Path(".").resolve()
BASE.mkdir(parents=True, exist_ok=True)
REPO = BASE / "ranked-feed"
OUT.mkdir(parents=True, exist_ok=True)
print({"mode": MODE, "kaggle": KAGGLE, "out": str(OUT), "free_gb": round(shutil.disk_usage(BASE).free / 1e9, 1)})
run("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv")


In [ ]:
# the code (public repository) and the packages: Qwen3-TTS without touching this machine's torch, Whisper for the check
if not REPO.exists():
    run("git", "clone", "-q", "--depth", "1", "--branch", "claude/tiny-speech-bilingual-v7lyp0", "https://github.com/larsboesen-prog/ranked-feed.git", REPO)
version = subprocess.run(["git", "-C", str(REPO), "log", "-1", "--format=%h %s"], capture_output=True, text=True, check=True).stdout
(OUT / "code-version.txt").write_text(version)
print(version)
GPU = REPO / "tiny-speech/training/gpu"
run(sys.executable, "-m", "pip", "install", "-q", "--no-deps", "qwen-tts==0.1.1")
run(sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.3", "accelerate==1.12.0", "librosa", "soundfile", "sox", "onnxruntime",
    "einops", "faster-whisper", "jiwer", "num2words")


In [ ]:
# this MODE's work
if MODE == "design":
    run(sys.executable, GPU / "design.py", "--out", OUT / "design")
else:
    raise SystemExit(f"MODE {MODE!r}: not yet")


In [ ]:
# what came out
summary = OUT / "design/summary.json"
if summary.exists():
    s = json.loads(summary.read_text())
    print({k: v for k, v in s.items() if k != "candidates"})
    for c in s["candidates"]:
        print(c["id"], c["label"], "words wrong es", c.get("wer_es"), "en", c.get("wer_en"))
